# OR-Gate Neural Network Using PyTorch

This notebook implements the PyTorch example in slides 16?19 of **Week 3 day 2.pptx**. A single neuron learns the OR truth table using two input weights and one bias.

Run the cells in order using **Python (University)**. PyTorch is already installed in this environment. If using a different environment, run `%pip install torch` in a separate cell and restart the kernel before continuing.

## Step 1 ? Import Libraries and Create the Dataset

PyTorch tensors hold the inputs and expected outputs. `torch.nn` provides neural-network layers and loss functions; `torch.optim` provides optimizers.

In [1]:
# Import PyTorch and its neural-network and optimization tools.
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


In [2]:
# Each row contains the two input features, x1 and x2.
# Floating-point tensors are needed for gradient-based learning.
X = torch.tensor([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
], dtype=torch.float32)

# OR is 0 only when both inputs are 0.
# Shape (4, 1) matches the model's single output for each example.
y = torch.tensor([
    [0.],
    [1.],
    [1.],
    [1.]
], dtype=torch.float32)

print("Inputs:\n", X)
print("Expected outputs:\n", y)
print("Input shape:", tuple(X.shape))
print("Target shape:", tuple(y.shape))

Inputs:
 tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
Expected outputs:
 tensor([[0.],
        [1.],
        [1.],
        [1.]])
Input shape: (4, 2)
Target shape: (4, 1)


## Step 2 ? Create the Neural Network

`nn.Linear(2, 1)` learns two weights and one bias, calculating $z = x_1w_1 + x_2w_2 + b$. `nn.Sigmoid()` converts this weighted sum into a probability between 0 and 1.

`nn.Sequential` applies these layers in order. This has the same structure as the TensorFlow `Dense(1, activation="sigmoid")` example.

In [3]:
# Fix the seed before creating the model so repeated runs are reproducible.
torch.manual_seed(42)

# Two inputs feed one neuron, followed by a sigmoid activation.
model = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid()
)

print(model)
print("Number of learned parameters:", sum(p.numel() for p in model.parameters()))

Sequential(
  (0): Linear(in_features=2, out_features=1, bias=True)
  (1): Sigmoid()
)
Number of learned parameters: 3


## Step 3 ? Configure the Loss Function and Optimizer

Binary cross-entropy measures how far the predicted probabilities are from the binary targets. SGD updates the weights and bias to reduce this loss, using a learning rate of `0.1`.

PyTorch uses these objects inside an explicit training loop, rather than calling Keras `model.compile()` and `model.fit()`.

In [4]:
# BCELoss expects probabilities, so it is paired with the sigmoid layer.
loss_function = nn.BCELoss()

# Give SGD all the trainable parameters: the two weights and the bias.
optimizer = optim.SGD(model.parameters(), lr=0.1)

print("Loss function:", loss_function)
print("Optimizer: SGD, learning rate = 0.1")

Loss function: BCELoss()
Optimizer: SGD, learning rate = 0.1


## Step 4 ? Train the Neural Network

Each epoch processes all four examples together, matching the slides. The loop performs five operations:

1. Calculate predictions with a forward pass.
2. Calculate the loss against the expected outputs.
3. Clear gradients left from the previous epoch.
4. Use `loss.backward()` to calculate new gradients automatically.
5. Use `optimizer.step()` to update the parameters.

Gradients describe how changing each parameter affects the loss. PyTorch accumulates gradients, so clearing them before each backward pass is necessary.

In [5]:
epochs = 1000
loss_history = []
model.train()

for epoch in range(epochs):
    # Forward pass: calculate probabilities for all four input combinations.
    predictions = model(X)

    # Compare the probabilities with the correct OR-gate outputs.
    loss = loss_function(predictions, y)

    # Clear previous gradients before computing the current gradients.
    optimizer.zero_grad()

    # Backpropagation: calculate gradients for the weights and bias.
    loss.backward()

    # Apply one SGD update to the weights and bias.
    optimizer.step()

    # Store an ordinary number without retaining the computation graph.
    loss_history.append(loss.item())

# Recompute the loss after the final parameter update.
with torch.no_grad():
    final_loss = loss_function(model(X), y).item()

print(f"Training completed: {epochs} epochs")
print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss:   {final_loss:.4f}")

Training completed: 1000 epochs
Initial loss: 0.4913
Final loss:   0.0855


## Step 5 ? Check the OR Truth Table

Use `model.eval()` for evaluation and `torch.no_grad()` to avoid calculating gradients during prediction. Probabilities of `0.5` or higher become class `1`; lower probabilities become class `0`.

These are the same four examples used for training. Accuracy here checks whether the model learned the complete OR truth table; it is not a score on unseen test data.

In [6]:
model.eval()

# Prediction does not require a gradient computation graph.
with torch.no_grad():
    prediction_probabilities = model(X)
    binary_predictions = (prediction_probabilities >= 0.5).to(torch.int64)

print("PyTorch OR-gate predictions")
print("-" * 72)

for inputs, probability, prediction, expected in zip(
    X, prediction_probabilities, binary_predictions, y
):
    x1, x2 = inputs.tolist()
    predicted_class = prediction.item()
    expected_class = int(expected.item())
    result = "Correct" if predicted_class == expected_class else "Incorrect"
    print(
        f"{int(x1)} OR {int(x2)} | "
        f"Probability: {probability.item():.4f} | "
        f"Prediction: {predicted_class} | Expected: {expected_class} | {result}"
    )

# Average the four correct/incorrect results to calculate accuracy.
accuracy = (binary_predictions == y.to(torch.int64)).float().mean().item()
print("\nPredictions:", binary_predictions.flatten().tolist())
print(f"Truth-table accuracy: {accuracy * 100:.1f}%")

PyTorch OR-gate predictions
------------------------------------------------------------------------
0 OR 0 | Probability: 0.1785 | Prediction: 0 | Expected: 0 | Correct
0 OR 1 | Probability: 0.9307 | Prediction: 1 | Expected: 1 | Correct
1 OR 0 | Probability: 0.9304 | Prediction: 1 | Expected: 1 | Correct
1 OR 1 | Probability: 0.9988 | Prediction: 1 | Expected: 1 | Correct

Predictions: [0, 1, 1, 1]
Truth-table accuracy: 100.0%


## Step 6 ? Inspect the Learned Parameters

The linear layer stores its weights with shape `(1, 2)` and its bias with shape `(1,)`. Positive weights increase the output when either input is active. A negative bias helps keep `(0, 0)` below the classification threshold.

In [7]:
# The first layer in the sequential model is the linear neuron.
output_layer = model[0]

# detach() lets us inspect learned values without tracking gradients.
learned_weights = output_layer.weight.detach()
learned_bias = output_layer.bias.detach()

print("PyTorch learned parameters")
print("-" * 35)
print(f"Weight for x1: {learned_weights[0, 0].item():.4f}")
print(f"Weight for x2: {learned_weights[0, 1].item():.4f}")
print(f"Bias:          {learned_bias[0].item():.4f}")

PyTorch learned parameters
-----------------------------------
Weight for x1: 4.1200
Weight for x2: 4.1233
Bias:          -1.5264


## Step 7 ? Compare the Three Implementations

| Operation | From scratch | TensorFlow / Keras | PyTorch |
|---|---|---|---|
| Store inputs | NumPy arrays | NumPy arrays | `torch.tensor` |
| Create weights and bias | Initialize manually | `Dense(1)` | `nn.Linear(2, 1)` |
| Calculate weighted sum | `np.dot(X, weights) + bias` | Dense layer | Linear layer |
| Apply activation | Binary step function | Sigmoid activation | `nn.Sigmoid()` |
| Measure error | `target - prediction` | Binary cross-entropy | `nn.BCELoss()` |
| Calculate updates | Perceptron update rule | Automatic differentiation | `loss.backward()` |
| Update parameters | Manual updates | SGD optimizer | `optimizer.step()` |
| Repeat training | Manual epoch loop | `model.fit()` | Explicit epoch loop |
| Predict | Custom `predict()` | `model.predict()` | `model(X)` under `torch.no_grad()` |

The original perceptron uses a step activation and corrections based on classification mistakes. The TensorFlow and PyTorch models use a differentiable sigmoid and minimize binary cross-entropy, so their learning rules differ even though all can learn OR.

PyTorch exposes the forward pass, gradient calculation and parameter update in the training loop. The learned probabilities can differ from TensorFlow because initialization and training settings differ; the expected binary predictions remain `[0, 1, 1, 1]`.

References: slides 16?19 of *Week 3 day 2.pptx*; [PyTorch training example](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html); [BCELoss documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html).